# Reference Implementation

This notebook contains reference implementations using `scikit-learn` for the Decision Tree (M1) and Neural Network Regressor (M2) models, matching the hyperparameters of the scratch implementations.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

# Add src to path
sys.path.append('src')

from src.data import load_and_preprocess_data
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
# Load data
train_path = 'claims_train.csv'
test_path = 'claims_test.csv'

# Check if files exist, if not assume we are in project root relative to them
if not os.path.exists(train_path):
    # Attempt to locate them if the notebook is run from a different CWD
    # But for now assuming run from project root
    print(f"Warning: {train_path} not found.")

df_train, df_test, X_train, X_test, y_train, y_test, X_train_std, X_test_std = load_and_preprocess_data(train_path, test_path)

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

## M1: Decision Tree Regressor (Reference)

Matching hyperparameters:
- `max_depth=14`
- `min_samples_split=200`
- `min_samples_leaf=10`

In [ ]:
dt_ref = DecisionTreeRegressor(
    max_depth=14,
    min_samples_split=200,
    min_samples_leaf=10,
    random_state=42
)

dt_ref.fit(X_train, y_train)

y_train_pred_dt = dt_ref.predict(X_train)
y_test_pred_dt = dt_ref.predict(X_test)

print("M1 Reference (sklearn DecisionTree):")
print(f"Train RMSE: {rmse(y_train, y_train_pred_dt):.4f}")
print(f"Test  RMSE: {rmse(y_test, y_test_pred_dt):.4f}")

## M2: Neural Network Regressor (Reference)

Matching hyperparameters:
- Hidden Layer: 1 layer, 32 units (`hidden_layer_sizes=(32,)`)
- Activation: ReLU
- Solver: SGD
- Learning Rate: 0.01 (`learning_rate_init=0.01`)
- Batch Size: 2048
- L2 Penalty: 1e-4 (`alpha=1e-4`)
- Epochs: 80 (`max_iter=80`)

In [ ]:
# Note: sklearn's SGD might have different defaults for momentum/nesterov than the scratch simple SGD.
# We'll set momentum to 0 to try match simple SGD if that's what the scratch code did,
# but looking at the scratch code it implements vanilla SGD (Momentum not explicitly mentioned/used).
# Scratch: self.W -= self.lr * grad. This is vanilla SGD.
# Sklearn MLPRegressor with solver='sgd' uses momentum=0.9 by default. We set it to 0.

nn_ref = MLPRegressor(
    hidden_layer_sizes=(32,),
    activation='relu',
    solver='sgd',
    learning_rate_init=0.01,
    max_iter=80,
    batch_size=2048,
    alpha=1e-4,          # L2 regularization
    momentum=0.0,        # Vanilla SGD to match scratch implementation
    random_state=42,
    learning_rate='constant', # Keep learning rate constant
    n_iter_no_change=80, # Prevent early stopping to force 80 epochs like the scratch one
)

nn_ref.fit(X_train_std, y_train)

y_train_pred_nn = nn_ref.predict(X_train_std)
y_test_pred_nn = nn_ref.predict(X_test_std)

print("M2 Reference (sklearn MLPRegressor):")
print(f"Train RMSE: {rmse(y_train, y_train_pred_nn):.4f}")
print(f"Test  RMSE: {rmse(y_test, y_test_pred_nn):.4f}")